# 08 - Move the data from Drive to Ibex

Ibex sits inside the KAUST network, unreachable from Colab - but Ibex itself can reach Google. So this notebook runs **on Ibex** and pulls the processed StarX data straight from your Google Drive with rclone:

1. get the rclone binary (module or a user-local download, no root),
2. authorize Drive access once, using your laptop's browser and a pasted token - read-only scope,
3. pull the selected directories (shards, smoke shards, run checkpoints, stats); every pull is resumable,
4. verify counts, then follow the last section to run notebook 05 on a GPU node.

Where to run it: clone the repo on Ibex (`git clone https://github.com/SattamAltwaim/StarX.git`) and open this notebook in a Jupyter session, or simply run the same lines in a terminal - every cell is plain commands, and a `tmux` on the login node works just as well for a pure transfer. No GPU needed.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import json
import os
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

REMOTE = "gdrive"                 # rclone remote name for your Google Drive
DRIVE_PATH = "StarX"              # the StarX folder at the top of My Drive
IBEX_DEST = f"/ibex/user/{os.environ.get('USER', 'unknown')}/StarX"

INCLUDE_SHARDS = True      # shards/train + shards/test (the training data)
INCLUDE_SMOKE = True       # the 20-design smoke shards (small)
INCLUDE_RUNS = True        # checkpoints, logs, val grids of existing runs
INCLUDE_STATS = True       # splits.json and dataset stats (tiny)
INCLUDE_RAW_ZIP = False    # the 2 GB dataset zip - only needed for
                           # notebooks 01-03 and 06's GT meshes on Ibex

Path(IBEX_DEST).mkdir(parents=True, exist_ok=True)
print(f"Drive '{DRIVE_PATH}' (remote '{REMOTE}')  ->  {IBEX_DEST}")

In [ ]:
# Make sure rclone is available: use the system/module one if present,
# otherwise download the official self-contained binary into ~/bin
# (no root needed).
RCLONE = shutil.which("rclone")
if RCLONE is None:
    bin_dir = Path.home() / "bin"
    RCLONE = str(bin_dir / "rclone")
    if not Path(RCLONE).exists():
        print("downloading the rclone binary...")
        bin_dir.mkdir(parents=True, exist_ok=True)
        archive = Path.home() / "rclone-current-linux-amd64.zip"
        urllib.request.urlretrieve(
            "https://downloads.rclone.org/rclone-current-linux-amd64.zip",
            archive,
        )
        with zipfile.ZipFile(archive) as zf:
            member = next(n for n in zf.namelist() if n.endswith("/rclone"))
            Path(RCLONE).write_bytes(zf.read(member))
        Path(RCLONE).chmod(0o755)
        archive.unlink()

version = subprocess.run(
    [RCLONE, "version"], capture_output=True, text=True
).stdout.splitlines()[0]
print(f"using {RCLONE}  ({version})")

## One-time Drive authorization

Ibex has no browser, so the Google sign-in happens on your laptop and the resulting token gets pasted here:

1. on your laptop: `brew install rclone` (macOS; any OS works),
2. run `rclone authorize "drive"` - a browser window opens; sign in with the Google account that holds the StarX folder and approve read access,
3. the terminal then prints a token: a single JSON blob between `--->` markers, starting with `{"access_token":`. Copy the whole blob,
4. run the next cell and paste it at the prompt.

This is needed once per Ibex account - the token is stored in your rclone config and refreshes itself afterwards.

In [ ]:
# Create the Drive remote (skipped if it already exists). Paste the token
# JSON from the previous step when prompted - getpass keeps it out of the
# notebook output. The remote is created read-only: Ibex can download
# from your Drive but never modify it.
import getpass

existing = subprocess.run(
    [RCLONE, "listremotes"], capture_output=True, text=True
).stdout
if f"{REMOTE}:" in existing:
    print(f"remote '{REMOTE}' already configured - nothing to do")
else:
    token_json = getpass.getpass(
        "paste the token JSON from 'rclone authorize \"drive\"': "
    ).strip()
    created = subprocess.run(
        [RCLONE, "config", "create", REMOTE, "drive",
         "token", token_json, "scope", "drive.readonly"],
        capture_output=True, text=True,
    )
    if created.returncode != 0:
        print(created.stderr)
        raise RuntimeError(
            "remote creation failed - fallback: run 'rclone config' in an "
            "Ibex terminal, choose a new 'drive' remote, and answer 'n' to "
            "the auto-config question to get the same paste-a-token flow"
        )
    print(f"remote '{REMOTE}' created (read-only scope)")

In [ ]:
# Sanity check: the remote works and the StarX folder is visible.
listing = subprocess.run(
    [RCLONE, "lsd", f"{REMOTE}:{DRIVE_PATH}"], capture_output=True, text=True
)
print(listing.stdout)
if listing.returncode != 0:
    print(listing.stderr)
    raise RuntimeError(
        f"could not list {DRIVE_PATH} on Drive - check the remote "
        "(previous cell) and that the folder name matches your Drive"
    )
print("Drive remote is working")

In [ ]:
# Inventory: what will be pulled, per the toggles above, sized by asking
# Drive directly.
SELECTION = {
    "shards/train": INCLUDE_SHARDS,
    "shards/test": INCLUDE_SHARDS,
    "shards/smoke_train": INCLUDE_SMOKE,
    "shards/smoke_test": INCLUDE_SMOKE,
    "shards/smoke": INCLUDE_SMOKE,
    "stats": INCLUDE_STATS,
    "runs": INCLUDE_RUNS,
    "raw": INCLUDE_RAW_ZIP,
}

transfer_dirs = []
total_bytes = 0
print(f"{'directory':24s} {'files':>7s} {'size':>10s}  status")
for rel, include in SELECTION.items():
    if not include:
        print(f"{rel:24s} {'-':>7s} {'-':>10s}  skipped (toggle off)")
        continue
    probe = subprocess.run(
        [RCLONE, "size", f"{REMOTE}:{DRIVE_PATH}/{rel}", "--json"],
        capture_output=True, text=True,
    )
    if probe.returncode != 0:
        print(f"{rel:24s} {'-':>7s} {'-':>10s}  absent on Drive")
        continue
    stats = json.loads(probe.stdout)
    if stats["count"] == 0:
        print(f"{rel:24s} {'-':>7s} {'-':>10s}  empty on Drive")
        continue
    total_bytes += stats["bytes"]
    transfer_dirs.append(rel)
    print(f"{rel:24s} {stats['count']:>7d} {stats['bytes'] / 2**30:>9.2f}G  queued")
print(f"\ntotal to pull: {total_bytes / 2**30:.2f} GiB "
      f"across {len(transfer_dirs)} directories")

In [ ]:
# The pull. rclone copy is resumable and skips files that already match,
# so killing and re-running this cell is always safe. KAUST's link to
# Google is fast; the whole set usually lands in minutes.
transferred = []
for rel in transfer_dirs:
    print(f"\n=== {rel} ===")
    result = subprocess.run(
        [RCLONE, "copy", f"{REMOTE}:{DRIVE_PATH}/{rel}", f"{IBEX_DEST}/{rel}",
         "--transfers", "8", "--checkers", "8",
         "--stats-one-line", "--stats", "10s", "-v"],
    )
    if result.returncode == 0:
        transferred.append(rel)
        print(f"done: {rel}")
    else:
        print(f"rclone exited with {result.returncode} for {rel} - "
              "re-run this cell to resume")
print(f"\ncompleted {len(transferred)}/{len(transfer_dirs)} directories")

In [ ]:
# Verify: file counts on Ibex must match what Drive reports.
print(f"{'directory':24s} {'drive files':>12s} {'ibex files':>11s} {'ibex size':>10s}")
all_match = True
for rel in transferred:
    drive_count = subprocess.run(
        [RCLONE, "size", f"{REMOTE}:{DRIVE_PATH}/{rel}", "--json"],
        capture_output=True, text=True,
    )
    n_drive = json.loads(drive_count.stdout)["count"] if drive_count.returncode == 0 else "?"
    local_dir = Path(IBEX_DEST) / rel
    local_files = [f for f in local_dir.rglob("*") if f.is_file()]
    size_gib = sum(f.stat().st_size for f in local_files) / 2**30
    match = str(n_drive) == str(len(local_files))
    all_match = all_match and match
    marker = "" if match else "   MISMATCH - re-run the transfer cell"
    print(f"{rel:24s} {n_drive!s:>12s} {len(local_files):>11d} {size_gib:>9.2f}G{marker}")
print("\nall counts match - transfer complete" if all_match
      else "\nre-run the transfer cell, it only fetches what is missing")

## Running notebook 05 on Ibex

The data now sits where the notebooks expect it once you add one symlink (off-Colab, they resolve their storage root to `<repo>/data/StarX`):

```
mkdir -p $HOME/StarX/data
ln -s /ibex/user/$USER/StarX  $HOME/StarX/data/StarX
```

Environment: activate a conda/venv that has a CUDA build of PyTorch - the setup cell pip-installs everything else pinned in `starx/pins.py` into that same environment. Then start Jupyter on a GPU node, for example:

```
srun --time=6:00:00 --gres=gpu:a100:1 --cpus-per-task=8 --mem=64G --pty bash
conda activate <your-torch-env>
cd $HOME/StarX/experiments
jupyter lab --no-browser --ip=$(hostname -i)
```

(check the KAUST Ibex documentation for the currently recommended Jupyter workflow and partition names; `--gres=gpu:v100:1` also works - the notebook picks fp16 there and bf16 on A100 automatically.)

In notebook 05, run the cells top to bottom through the resume cell, then the post-mortem cells (loss curves, gallery, before/after, probe) to analyze the existing `baseline_l4` run before launching any new training.

## Cleaning up

The Drive token grants read-only access and lives in `~/.config/rclone/rclone.conf` on Ibex. When you no longer need transfers:

```
rclone config delete gdrive
```

and optionally revoke rclone's access under Google Account > Security > Third-party access.